<a href="https://colab.research.google.com/github/abhijadhav14/Data-Analytics-Using-Python/blob/main/pipeclassifi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create Spark Session
spark = SparkSession.builder \
    .appName("IrisClassificationPipeline") \
    .getOrCreate()

df = spark.read.csv("/content/drive/MyDrive/data/Iris.csv",header=True,inferSchema=True)

print("Original Dataset")
df.show(5)

# Split dataset into training and testing
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Stage 1: Convert Species into numeric labels
label_indexer = StringIndexer(
    inputCol="Species",
    outputCol="label"
)

# Stage 2: Assemble feature columns
assembler = VectorAssembler(
    inputCols=[
        "SepalLengthCm",
        "SepalWidthCm",
        "PetalLengthCm",
        "PetalWidthCm"
    ],
    outputCol="features"
)

# Stage 3: Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="label"
)

# Create Pipeline
pipeline = Pipeline(stages=[
    label_indexer,
    assembler,
    dt_classifier
])

# Train the pipeline model
pipeline_model = pipeline.fit(train_data)

# Make predictions
predictions = pipeline_model.transform(test_data)

# Display predictions
print("Prediction Results")
predictions.select(
    "Species",
    "label",
    "prediction"
).show()

# Evaluate model accuracy
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)

print("Model Accuracy =", accuracy)

# Stop Spark Session
spark.stop()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original Dataset
+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows
Prediction Results
+---------------+-----+----------+
|        Species|label|prediction|
+---------------+-----+----------+
|    Iris-setosa|  2.0|       2.0|
|    Iris-setosa|  2.0|